# Import and Setup

In [1]:
import os
import json
import ast
from openai import OpenAI

openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key
client = OpenAI()
MODEL = "gpt-5.4"

# LLM Functions

In [2]:
REQUIREMENTS = """- Must run quickly on a MacBook with 36GB RAM (Apple Silicon); use device='mps' where supported
- Use a single small transformer-based model (e.g. distilbert, all-MiniLM-L6-v2, or similarly lightweight models via transformers.pipeline or sentence-transformers)
- No training, fine-tuning, or weight updates — load a pretrained model and evaluate it directly (zero-shot or task-specific pretrained checkpoint)
- No hyperparameter tuning or loops over multiple models/configurations — pick one and run it
- Only one model and one dataset/subset
- Only code cells (no markdown cells)
- No plots or visualizations"""

In [3]:
def generate_data_science_tasks(n: int = 10) -> list:
    prompt = f"""Brainstorm a list of {n} descriptions of AI tasks that can be evaluated using a modern AI model and HuggingFace datasets.

Requirements for each task:
{REQUIREMENTS}
- Restrict to tasks with datasets that have less than a million samples
- Each description should specify both the task type and the dataset

Return ONLY a valid Python list of strings, no explanation."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    return ast.literal_eval(raw)


tasks = generate_data_science_tasks(10)
tasks

=== generate_data_science_tasks output ===
[
  "Sentiment classification on the IMDb movie reviews dataset using a pretrained DistilBERT sentiment-analysis pipeline evaluated directly on a small test split with device='mps' where supported.",
  "Natural language inference on the GLUE MNLI matched validation set using a pretrained lightweight BERT-style sequence-classification checkpoint run zero-shot without any training or tuning.",
  "Paraphrase detection on the GLUE MRPC dataset using a pretrained sentence-transformers all-MiniLM-L6-v2 encoder with cosine similarity evaluated on the validation split.",
  "Semantic textual similarity on the STS-B validation set from GLUE using all-MiniLM-L6-v2 sentence embeddings and direct cosine similarity scoring.",
  "Extractive question answering on the SQuAD validation set using a small pretrained DistilBERT question-answering pipeline with direct evaluation on a limited subset for speed.",
  "Zero-shot topic classification on the AG News test 

["Sentiment classification on the IMDb movie reviews dataset using a pretrained DistilBERT sentiment-analysis pipeline evaluated directly on a small test split with device='mps' where supported.",
 'Natural language inference on the GLUE MNLI matched validation set using a pretrained lightweight BERT-style sequence-classification checkpoint run zero-shot without any training or tuning.',
 'Paraphrase detection on the GLUE MRPC dataset using a pretrained sentence-transformers all-MiniLM-L6-v2 encoder with cosine similarity evaluated on the validation split.',
 'Semantic textual similarity on the STS-B validation set from GLUE using all-MiniLM-L6-v2 sentence embeddings and direct cosine similarity scoring.',
 'Extractive question answering on the SQuAD validation set using a small pretrained DistilBERT question-answering pipeline with direct evaluation on a limited subset for speed.',
 'Zero-shot topic classification on the AG News test set using a lightweight MNLI-based transformer zero

In [6]:
def generate_notebook(task: str, notebook_dir: str) -> str:
    # Step 0: Name the notebook
    name_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""Generate a short, descriptive filename for a Jupyter notebook about this AI task:

Task: {task}

Requirements:
- Use snake_case
- End with .ipynb
- Be concise (3-6 words)
- Return ONLY the filename, nothing else."""}],
    )
    notebook_name = name_response.choices[0].message.content.strip()
    if not notebook_name.endswith(".ipynb"):
        notebook_name += ".ipynb"
    print(f"=== [generate_notebook] Step 0: Name ===\n{notebook_name}\n")

    # Step 1: Plan
    plan_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Plan a Jupyter notebook workflow for this AI task:

Task: {task}

Requirements:
{REQUIREMENTS}

Write a concise step-by-step plan for the notebook that respects all requirements above."""}],
    )
    plan = plan_response.choices[0].message.content.strip()
    print(f"=== [generate_notebook] Step 1: Plan ===\n{plan}\n")

    # Step 2: Generate notebook JSON
    nb_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Generate a complete Jupyter notebook as valid JSON for this AI task.

Task: {task}

Plan:
{plan}

Requirements:
{REQUIREMENTS}
- Use HuggingFace datasets to load data
- Include cells for imports, data loading, inference, and evaluation
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}],
    )
    raw = nb_response.choices[0].message.content.strip()
    print(f"=== [generate_notebook] Step 2: Raw notebook JSON (first 500 chars) ===\n{raw[:500]}\n")

    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.rsplit("```", 1)[0].strip()

    nb = json.loads(raw)
    os.makedirs(notebook_dir, exist_ok=True)
    path = os.path.join(notebook_dir, notebook_name)
    with open(path, "w") as f:
        json.dump(nb, f, indent=1)
    print(f"Notebook saved to {path}")
    return path

In [7]:
def generate_n_variations(notebook_path: str, n: int = 3) -> list:
    with open(notebook_path, "r") as f:
        original_nb = json.load(f)

    cells_text = []
    for cell in original_nb.get("cells", []):
        source = "".join(cell.get("source", []))
        if source.strip():
            cells_text.append(source)
    original_content = "\n\n---\n\n".join(cells_text)

    original_name = os.path.splitext(os.path.basename(notebook_path))[0]
    notebook_dir = os.path.dirname(notebook_path)
    variations_dir = os.path.join(notebook_dir, f"variations_{original_name}")
    os.makedirs(variations_dir, exist_ok=True)

    # Step 0: Plan all variations upfront, returning a name -> description dict
    plan_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Plan {n} variations of the following Jupyter notebook.

Original notebook:
{original_content}

Requirements that every variation must satisfy:
{REQUIREMENTS}
- Stay on the same task type as the original notebook (e.g. if it is emotion classification, all variations must also be emotion classification)
- Do not switch to a different task type

Return ONLY a valid Python dictionary mapping a snake_case name to a concise description for each variation, no explanation. The name will be the file name for the notebook without extensions."""}],
    )
    raw_plans = plan_response.choices[0].message.content.strip()
    print(f"=== [generate_n_variations] Step 0: Variation plans for '{original_name}' ===\n{raw_plans}\n")
    variation_plans = ast.literal_eval(raw_plans)

    paths = []
    for name, plan in variation_plans.items():
        print(f"--- [generate_n_variations] Generating variation: '{name}' ---")
        print(f"Plan: {plan}\n")

        # Step 1: Generate the variation notebook according to its plan
        nb_response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"""You are an expert data scientist. Generate a Jupyter notebook as valid JSON implementing this variation of an existing notebook.

Original notebook:
{original_content}

Variation plan:
{plan}

Requirements:
{REQUIREMENTS}
- Use HuggingFace datasets to load data
- Include cells for imports, data loading, inference, and evaluation
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}],
        )
        raw = nb_response.choices[0].message.content.strip()
        print(f"=== [generate_n_variations] Raw notebook JSON for '{name}' (first 300 chars) ===\n{raw[:300]}\n")

        if raw.startswith("```"):
            raw = raw.split("```", 2)[1]
            if raw.startswith("json"):
                raw = raw[4:]
            raw = raw.rsplit("```", 1)[0].strip()

        nb = json.loads(raw)
        variation_name = f"{name}.ipynb"
        path = os.path.join(variations_dir, variation_name)
        with open(path, "w") as f:
            json.dump(nb, f, indent=1)
        print(f"Variation saved to {path}")
        paths.append(path)

    return paths

# Initial Dataset

- Select best suited task from list
- Create initial notebook and store in "../notebooks/batch_2" folder.
- Create 5 variations of the initial notebook.
- For each variation, create 2-4 new variations of the variation.

In [8]:
selected_task = 'Text classification for emotion detection on the dair-ai/emotion dataset test split using a small pretrained DistilBERT emotion classifier and straightforward accuracy evaluation.'

In [9]:
import shutil

NOTEBOOK_DIR = "../notebooks/batch_3"
if os.path.exists(NOTEBOOK_DIR):
    shutil.rmtree(NOTEBOOK_DIR)
os.makedirs(NOTEBOOK_DIR)

notebook_path = generate_notebook(selected_task, NOTEBOOK_DIR)

=== [generate_notebook] Step 0: Name ===
distilbert_emotion_classification.ipynb

=== [generate_notebook] Step 1: Plan ===
```python
# 1) Install/import dependencies
# - Install only what's needed: transformers, datasets, torch, scikit-learn
# - Import pandas/numpy optionally for quick inspection
# - Import pipeline from transformers, load_dataset from datasets, and accuracy_score/classification_report from sklearn

# 2) Set runtime/device for Apple Silicon
# - Detect MPS availability with torch.backends.mps.is_available()
# - Set device = "mps" if available, otherwise "cpu"
# - Print selected device
# - Keep batch size modest for fast inference on a MacBook

# 3) Load the dataset
# - Load only the test split of dair-ai/emotion via datasets.load_dataset("dair-ai/emotion", split="test")
# - Inspect a few rows and dataset size
# - Note that labels are integers 0..5 with known class names:
#   ["sadness", "joy", "love", "anger", "fear", "surprise"]

# 4) Choose exactly one small pretraine

In [10]:
variation_paths = generate_n_variations(notebook_path, n=5)

=== [generate_n_variations] Step 0: Variation plans for 'distilbert_emotion_classification' ===
{
    "emotion_zero_shot_bart_go_emotions_subset": "Evaluate a lightweight zero-shot transformer classifier on a small GoEmotions subset by mapping candidate emotion labels to the six canonical classes and reporting accuracy and a classification report.",
    "emotion_sentence_transformer_prototype_matching": "Use a compact sentence-transformer to encode test texts and six class prototype phrases, assign each text to the nearest emotion by cosine similarity, and evaluate on the dair-ai/emotion test split.",
    "emotion_distilbert_validation_split_audit": "Run a small pretrained emotion classification pipeline on the dair-ai/emotion validation split, normalize model labels to canonical classes, and report metrics plus a compact error table.",
    "emotion_distilroberta_emotion_test_evaluation": "Swap in a lightweight pretrained emotion-specific transformer checkpoint, evaluate directly on th

In [11]:
for vp in variation_paths:
    generate_n_variations(vp, n=5)

=== [generate_n_variations] Step 0: Variation plans for 'emotion_zero_shot_bart_go_emotions_subset' ===
{
    "emotion_zero_shot_mnli_balanced_goemotions": "Use a lightweight MNLI zero-shot classifier on a class-balanced six-emotion GoEmotions subset, evaluate accuracy and classification report on MPS when available.",
    "emotion_feature_extraction_label_embedding_similarity": "Use a small sentence-transformer or feature-extraction model to embed texts and emotion label prompts, classify by cosine similarity on a six-class GoEmotions subset.",
    "emotion_text_classification_twitter_roberta_mapped_goemotions": "Use a pretrained emotion classification checkpoint with six emotion outputs aligned or mapped to the notebook labels, then evaluate directly on a mapped GoEmotions test subset.",
    "emotion_zero_shot_short_text_subset_goemotions": "Use a small zero-shot transformer on a short-text-only GoEmotions subset to keep runtime low, then report predictions and standard classificatio

# Dataset Variation Test

In [12]:
selected_task = "Paraphrase detection on the GLUE MRPC dataset using a pretrained sentence-pair classification model such as DistilBERT, evaluated without any training."

In [13]:
import shutil

NOTEBOOK_DIR = "../notebooks/batch_4"
if os.path.exists(NOTEBOOK_DIR):
    shutil.rmtree(NOTEBOOK_DIR)
os.makedirs(NOTEBOOK_DIR)

notebook_path = generate_notebook(selected_task, NOTEBOOK_DIR)

=== [generate_notebook] Step 0: Name ===
mrpc_paraphrase_inference.ipynb

=== [generate_notebook] Step 1: Plan ===
```python
# 1) Install/import only the minimal libraries needed.
#    - Use: datasets, transformers, torch, scikit-learn, pandas
#    - Keep everything lightweight and notebook-friendly on Apple Silicon.
```

```python
# 2) Set up device selection with priority for Apple Silicon MPS.
#    - Detect torch.backends.mps.is_available()
#    - Use device = torch.device("mps") if available else "cpu"
#    - Print selected device
```

```python
# 3) Fix the single model choice up front.
#    - Use one small pretrained sentence-pair classification checkpoint only
#    - Recommended: "textattack/distilbert-base-uncased-MRPC"
#      because it is already fine-tuned for MRPC and can be evaluated directly
#    - Do not compare against any other model
```

```python
# 4) Load tokenizer and model once.
#    - AutoTokenizer.from_pretrained(...)
#    - AutoModelForSequenceClassification.fr

In [14]:
variation_paths = generate_n_variations(notebook_path, n=5)

=== [generate_n_variations] Step 0: Variation plans for 'mrpc_paraphrase_inference' ===
{
    "mrpc_pipeline_evaluation": "Use transformers.pipeline for text-pair classification on GLUE MRPC validation with MPS if available, then compute accuracy, precision, recall, F1, confusion matrix, and show mismatches.",
    "mrpc_manual_batched_inference": "Keep direct tokenizer and AutoModelForSequenceClassification inference on GLUE MRPC validation, but streamline batching, add max_length for faster tokenization, and summarize metrics plus sample errors.",
    "mrpc_sentence_transformer_cross_encoder": "Evaluate a lightweight sentence-transformers CrossEncoder pretrained for MS MARCO or paraphrase-style pair scoring on the MRPC validation split, convert scores to binary predictions, and report classification metrics.",
    "mrpc_subset_quick_check": "Run the same pretrained small sequence-classification model on a fixed small subset of GLUE MRPC validation for a very fast sanity-check notebook

In [15]:
for vp in variation_paths:
    generate_n_variations(vp, n=5)

=== [generate_n_variations] Step 0: Variation plans for 'mrpc_pipeline_evaluation' ===
{
    "mrpc_distilbert_pipeline_baseline": "Baseline MRPC paraphrase classification notebook using a lightweight Hugging Face text-classification pipeline with MPS support, batched validation inference, standard binary metrics, prediction table, mismatches, and runtime summary.",
    "mrpc_distilbert_manual_tokenization": "MRPC paraphrase classification notebook using AutoTokenizer and AutoModelForSequenceClassification directly on MPS for manual batched inference, then computing accuracy, precision, recall, F1, confusion matrix, and mismatch inspection.",
    "mrpc_distilbert_subset_quick_eval": "Fast MRPC paraphrase classification notebook evaluating a fixed small validation subset with a lightweight pretrained DistilBERT checkpoint on MPS, reporting binary metrics, compact results dataframe, sample predictions, and errors.",
    "mrpc_distilbert_confidence_analysis": "MRPC paraphrase classificatio